In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))
print(sys.path)

['/home/codespace/.python/current/lib/python312.zip', '/home/codespace/.python/current/lib/python3.12', '/home/codespace/.python/current/lib/python3.12/lib-dynload', '', '/workspaces/tastetester/.venv/lib/python3.12/site-packages', '/workspaces/tastetester']


In [1]:
import json

import pandas as pd
from etl.common.vectorstore import ListeningVectorStore

store = ListeningVectorStore()
tracks = store.execute("SELECT * FROM track_details").fetchall()
df_tracks = pd.DataFrame.from_records(tracks, columns=tracks[0].keys(), index="spotify_track_uri")
df_tracks['genres'] = df_tracks['genres'].apply(lambda x: json.loads(x) or [])
df_tracks['top_genre'] = df_tracks['genres'].apply(lambda x: x[0] if len(x) > 0 else None)
display(df_tracks.head())

df_tracks['total_plays'].hist(by=df_tracks["top_genre"], figsize=(20, 20))

ModuleNotFoundError: No module named 'etl'

In [25]:
from app.recommender import Recommender
import logging

logging.getLogger().setLevel(logging.DEBUG)

track = "09eSdS5RTgyodJt3krr5AC"
recommendations = Recommender().analyze(track)
print(recommendations)

DEBUG:root:Loading vector store
DEBUG:root:Fetching track details for 09eSdS5RTgyodJt3krr5AC
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.reccobeats.com:443
DEBUG:urllib3.connectionpool:https://api.reccobeats.com:443 "GET /v1/track?ids=09eSdS5RTgyodJt3krr5AC HTTP/1.1" 200 None
DEBUG:root:Got details {'id': '27736c03-57ac-40ba-b6ba-b2971f2cee25', 'trackTitle': 'Answer', 'artists': [{'id': 'a5223065-e579-44e9-8fb5-e11508acfa49', 'name': 'Phantogram', 'href': 'https://open.spotify.com/artist/1l9d7B8W0IHy3LqWsxP2SH'}], 'durationMs': 231600, 'isrc': 'USUM71605177', 'ean': None, 'upc': None, 'href': 'https://open.spotify.com/track/09eSdS5RTgyodJt3krr5AC', 'availableCountries': 'AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,DE,EC,EE,SV,FI,FR,GR,GT,HN,HK,HU,IS,IE,IT,LV,LT,LU,MY,MT,MX,NL,NZ,NI,NO,PA,PY,PE,PH,PL,PT,SG,SK,ES,SE,CH,TW,TR,UY,US,GB,AD,LI,MC,ID,JP,TH,VN,RO,IL,ZA,SA,AE,BH,QA,OM,KW,EG,MA,DZ,TN,LB,JO,PS,IN,BY,KZ,MD,UA,AL,BA,HR,ME,MK,RS,SI,KR,BD,PK,LK,GH,KE,NG,TZ,UG

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:track_name: object, artist_name: object, album_name: object, genres: object, top_genre: object

In [13]:
distances = store.execute("SELECT entity_key, vec_distance_cosine(vector, (SELECT vector FROM track_content_vectors LIMIT 1)) AS distance FROM track_content_vectors").fetchall()
df_distances = pd.DataFrame.from_records(distances, columns=distances[0].keys())

df_distances['distance'].describe()

count    5.340000e+02
mean     9.612464e-01
std      3.231277e-01
min     -2.220446e-16
25%      7.432737e-01
50%      9.771225e-01
75%      1.202131e+00
max      1.646639e+00
Name: distance, dtype: float64

In [14]:
from sklearn.metrics.pairwise import pairwise_distances

vectors = store.execute("SELECT entity_key, vec_to_json(vector) AS vector FROM track_content_vectors").fetchall()
keys = store.execute("SELECT feature_columns FROM vector_registry").fetchone()

def parse_row(row):
  entity_key = row['entity_key']
  vector = json.loads(row['vector'])

  return [entity_key] + vector

columns = ['entity_key'] + json.loads(keys['feature_columns'])

df_vectors = pd.DataFrame.from_records(map(parse_row, vectors), columns=columns, index='entity_key')
df_vectors = df_vectors.dropna()
display(df_vectors.head())

df_pairwise_distances = pairwise_distances(df_vectors, metric='cosine')
df_pairwise_distances

,danceability,energy,tempo,acousticness,valence,instrumentalness,liveness,speechiness,loudness,genre:alternative rock,...,genre:progressive metal,genre:progressive rock,genre:psychedelic pop,genre:psychedelic rock,genre:rock,genre:space rock,genre:stoner metal,genre:stoner rock,genre:synthwave,genre:trip hop
entity_key,,,,,,,,,,,,,,,,,,,,,
spotify:track:00RYw42tKwXecTaVay3BKl,1.016082,1.168719,-0.824202,-0.470081,-0.440962,-1.482137,-0.021799,-0.221650,1.565109,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:01PW4iTCvh9cUHPF0eV7H3,-0.869723,1.141809,1.169956,-0.534343,1.127291,-1.482305,-0.641739,0.158056,1.227448,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:01iiEi9h8CQGUN2K1xBbTj,-0.153988,0.392817,-1.002272,-0.185808,-0.231862,-1.480911,-0.544093,-0.286683,0.518211,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:01s8hwUgHNbSL4mJtaEtNK,-1.336506,-0.347204,0.386224,-0.500894,-0.768870,0.880444,-0.419197,-0.337031,-0.125165,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:02FPjj5HyCRQMbE3egggh6,-1.212031,1.177689,0.307122,-0.526260,-0.988900,0.357728,-0.396488,-0.007672,1.259887,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


array([[0.        , 0.71346369, 0.50674901, ..., 0.83865728, 1.21317833,
        0.8930585 ],
       [0.71346369, 0.        , 0.74338603, ..., 1.17802029, 0.81260105,
        0.82114956],
       [0.50674901, 0.74338603, 0.        , ..., 1.02856062, 1.21269797,
        1.1935418 ],
       ...,
       [0.83865728, 1.17802029, 1.02856062, ..., 0.        , 0.62725264,
        0.78168201],
       [1.21317833, 0.81260105, 1.21269797, ..., 0.62725264, 0.        ,
        0.67590427],
       [0.8930585 , 0.82114956, 1.1935418 , ..., 0.78168201, 0.67590427,
        0.        ]], shape=(534, 534))

In [19]:
df_all = df_tracks.join(df_vectors.filter(like='genre:'), rsuffix="_")
display(df_all.head())

print(df_tracks.columns)
print(df_all.columns)

,track_name,artist_name,album_name,total_plays,total_playtime_ms,years_listened,most_recent_year,peak_year,danceability,energy,...,genre:progressive metal,genre:progressive rock,genre:psychedelic pop,genre:psychedelic rock,genre:rock,genre:space rock,genre:stoner metal,genre:stoner rock,genre:synthwave,genre:trip hop
spotify_track_uri,,,,,,,,,,,,,,,,,,,,,
spotify:track:00RYw42tKwXecTaVay3BKl,Howling At The Moon,Phantogram,Voices,1,238186,1,2017,2017.0,0.597,0.965,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:0129SdVdz3RBtafIMgN1pu,My opinion,deadmau5,stuff i used to do,1,248453,1,2017,2017.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spotify:track:01AwwfC3sYLbtmECslpj2O,Try again,deadmau5,stuff i used to do,4,689764,1,2017,2017.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spotify:track:01PW4iTCvh9cUHPF0eV7H3,Time Consumer,Coheed and Cambria,The Second Stage Turbine Blade (Re-Issue),2,683386,1,2018,2018.0,0.294,0.959,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:01iiEi9h8CQGUN2K1xBbTj,List Of People (To Try And Forget About),Tame Impala,Currents B-Sides & Remixes,1,279986,1,2019,2019.0,0.409,0.792,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


Index(['track_name', 'artist_name', 'album_name', 'total_plays',
       'total_playtime_ms', 'years_listened', 'most_recent_year', 'peak_year',
       'danceability', 'energy', 'tempo', 'acousticness', 'valence',
       'instrumentalness', 'liveness', 'speechiness', 'loudness', 'genres',
       'total_tracks', 'release_date_year', 'top_genre'],
      dtype='object')
Index(['track_name', 'artist_name', 'album_name', 'total_plays',
       'total_playtime_ms', 'years_listened', 'most_recent_year', 'peak_year',
       'danceability', 'energy', 'tempo', 'acousticness', 'valence',
       'instrumentalness', 'liveness', 'speechiness', 'loudness', 'genres',
       'total_tracks', 'release_date_year', 'top_genre',
       'genre:alternative rock', 'genre:art punk', 'genre:audiobook',
       'genre:author', 'genre:darksynth', 'genre:electronic', 'genre:folk',
       'genre:french', 'genre:glitch pop', 'genre:hard rock',
       'genre:has german audiobooks', 'genre:house', 'genre:indie pop',
     

In [20]:
from scipy import stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
import numpy as np

df_train = df_all.dropna()
l_all = len(df_all)
print(f"Total tracks in dataset: {l_all}")

total_plays = df_train['total_plays']

# z_scores = np.abs(stats.zscore(total_plays))
# z_threshold = 5
# df_train = df_train[z_scores < z_threshold]
# print(f"Total outliers removed: {l_all - len(df_train)}")

print(f"Average number of plays for tracks in dataset: {total_plays.mean()}")
print(f"Median number of plays for tracks in dataset: {total_plays.median()}")

X = df_train[df_vectors.columns].copy()
# X = X.drop(columns='genre:pop')
# X = pd.DataFrame(sm.add_constant(X))
display(X.head())

y = df_train['total_plays'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = sm.GLM(y_train, X_train, family=sm.families.Poisson())
results = model.fit(scale='x2')

print(results.summary())

from sklearn.metrics import mean_absolute_error, r2_score
preds = results.predict(X_test)
print('MAE:', mean_absolute_error(y_test.values, preds))
print('corrcoef:', np.corrcoef(y_test.values, preds)[0,1])
print('r^2:', r2_score(y_test.values, preds))

model = sm.GLM(y, X, family=sm.families.Poisson())
results = model.fit(scale='x2')


Total tracks in dataset: 792
Average number of plays for tracks in dataset: 5.252873563218391
Median number of plays for tracks in dataset: 2.0


,danceability,energy,tempo,acousticness,valence,instrumentalness,liveness,speechiness,loudness,genre:alternative rock,...,genre:progressive metal,genre:progressive rock,genre:psychedelic pop,genre:psychedelic rock,genre:rock,genre:space rock,genre:stoner metal,genre:stoner rock,genre:synthwave,genre:trip hop
spotify_track_uri,,,,,,,,,,,,,,,,,,,,,
spotify:track:00RYw42tKwXecTaVay3BKl,0.597,0.965,100.029,0.016600,0.1510,0.000116,0.1950,0.0446,-1.420,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:01PW4iTCvh9cUHPF0eV7H3,0.294,0.959,155.541,0.000437,0.4810,0.000053,0.0858,0.0627,-2.794,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:01iiEi9h8CQGUN2K1xBbTj,0.409,0.792,95.072,0.088100,0.1950,0.000576,0.1030,0.0415,-5.680,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:01s8hwUgHNbSL4mJtaEtNK,0.219,0.627,133.724,0.008850,0.0820,0.886000,0.1250,0.0391,-8.298,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
spotify:track:02FPjj5HyCRQMbE3egggh6,0.239,0.967,131.522,0.002470,0.0357,0.690000,0.1290,0.0548,-2.662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


                 Generalized Linear Model Regression Results                  
Dep. Variable:            total_plays   No. Observations:                  417
Model:                            GLM   Df Residuals:                      389
Model Family:                 Poisson   Df Model:                           27
Link Function:                    Log   Scale:                          5.6434
Method:                          IRLS   Log-Likelihood:                -259.17
Date:                Mon, 08 Jun 2026   Deviance:                       1716.7
Time:                        23:12:21   Pearson chi2:                 2.20e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.3188
Covariance Type:            nonrobust                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
danceability      

In [21]:
import xgboost as xgb
import optuna

def objective(trial):
  pruning_callback = optuna.integration.XGBoostPruningCallback(trial, 'validation_0-poisson-nloglik')
  params = {
    "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
    "max_depth": trial.suggest_int("max_depth", 6, 12),
    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
    "tree_method": trial.suggest_categorical("tree_method", ["exact", "approx", "hist"]),
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",
    "seed": 42,
    "callbacks": [pruning_callback],
    "early_stopping_rounds": 50
  }


  model = xgb.XGBRegressor(**params)
  model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
  results = model.evals_result()
  return np.max(results['validation_0']['poisson-nloglik'])


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50)

model = xgb.XGBRegressor(**study.best_params)
model.fit(X, y)
print(model.score(X, y))


[I 2026-06-08 23:12:32,819] A new study created in memory with name: no-name-c454811b-bd7e-4d90-bc5e-b45a2dcdedd4


[I 2026-06-08 23:12:35,011] Trial 0 finished with value: 7.794917903627668 and parameters: {'n_estimators': 437, 'max_depth': 12, 'learning_rate': 0.07587945476302646, 'tree_method': 'exact'}. Best is trial 0 with value: 7.794917903627668.
[I 2026-06-08 23:12:35,707] Trial 1 finished with value: 8.001934268361046 and parameters: {'n_estimators': 152, 'max_depth': 12, 'learning_rate': 0.0641003510568888, 'tree_method': 'hist'}. Best is trial 1 with value: 8.001934268361046.
[I 2026-06-08 23:12:35,879] Trial 2 finished with value: 6.819049058641706 and parameters: {'n_estimators': 850, 'max_depth': 7, 'learning_rate': 0.02636424704863906, 'tree_method': 'hist'}. Best is trial 1 with value: 8.001934268361046.
[I 2026-06-08 23:12:36,090] Trial 3 finished with value: 7.505321749051412 and parameters: {'n_estimators': 489, 'max_depth': 8, 'learning_rate': 0.06506676052501416, 'tree_method': 'hist'}. Best is trial 1 with value: 8.001934268361046.
[I 2026-06-08 23:12:36,584] Trial 4 finished w

0.8917219042778015


In [24]:
vector = recommendations['vector']

input = pd.DataFrame.from_records([vector], columns=store.feature_columns())
# input = input.drop(columns='genre:pop')
# input = pd.DataFrame(sm.add_constant(input))
# input = input.reindex(columns=X.columns, fill_value=0)
display(input)

expected_plays = model.predict(input)
print(f"Expected number of plays for {recommendations['details']['artists'][0]['name']} - {recommendations['details']['trackTitle']}: {expected_plays[0]:.2f}")

,danceability,energy,tempo,acousticness,valence,instrumentalness,liveness,speechiness,loudness,genre:alternative rock,...,genre:progressive metal,genre:progressive rock,genre:psychedelic pop,genre:psychedelic rock,genre:rock,genre:space rock,genre:stoner metal,genre:stoner rock,genre:synthwave,genre:trip hop
0,-0.446506,-1.495359,0.912568,-0.023195,-0.650063,-1.364836,-0.441905,-0.477585,-0.146053,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Expected number of plays for Phantogram - Answer: 8.00


In [11]:
df_tracks.sort_values(by='total_plays').tail(1)

,track_name,artist_name,album_name,total_plays,total_playtime_ms,years_listened,most_recent_year,peak_year,danceability,energy,...,acousticness,valence,instrumentalness,liveness,speechiness,loudness,genres,total_tracks,release_date_year,top_genre
spotify_track_uri,,,,,,,,,,,,,,,,,,,,,
spotify:track:5tVKO8f1v0aTBuOllPxqX0,Hunter Moon,Russian Circles,Blood Year,57,7397454,7,2025,2022.0,0.273,0.0575,...,0.0907,0.0595,0.851,0.0917,0.042,-24.132,"[math rock, post-metal, post-rock]",7.0,2019,math rock
